In [1]:
"""
test_trained_mlps.py -- test the trained per-surface MLPs on a new subject.

One model per head surface: mlp_cell01.pt ... mlp_cell20.pt. Each checkpoint
holds its own weights, its own feature normalisation (mean / std of the
training subjects) and its own decision threshold.

This script loads all of them, applies each one to the matching surface of the
test subject, prints the metrics and saves the predictions for MATLAB.

Run it with the paths in the CONFIG block below:

    python test_trained_mlps.py

or override any of them on the command line:

    python test_trained_mlps.py --model-dir Trained_models --test-dir Testing_dataset \
                                --subject HC023 --results-dir Testing_Results

Self-contained: the MLP class and the loaders are defined here, so no other
file is needed.
"""

'\ntest_trained_mlps.py -- test the trained per-surface MLPs on a new subject.\n\nOne model per head surface: mlp_cell01.pt ... mlp_cell20.pt. Each checkpoint\nholds its own weights, its own feature normalisation (mean / std of the\ntraining subjects) and its own decision threshold.\n\nThis script loads all of them, applies each one to the matching surface of the\ntest subject, prints the metrics and saves the predictions for MATLAB.\n\nRun it with the paths in the CONFIG block below:\n\n    python test_trained_mlps.py\n\nor override any of them on the command line:\n\n    python test_trained_mlps.py --model-dir Trained_models --test-dir Testing_dataset                                 --subject HC023 --results-dir Testing_Results\n\nSelf-contained: the MLP class and the loaders are defined here, so no other\nfile is needed.\n'

In [2]:
import argparse
import os

import numpy as np
import torch
import torch.nn as nn
from scipy.io import loadmat, savemat

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# ----------------------------------------------------------------- CONFIG
model_dir   = '/content/drive/MyDrive/Mesh_segmentation_new/Trained_models'
test_dir    = '/content/drive/MyDrive/Mesh_segmentation_new/Testing_dataset'
results_dir = '/content/drive/MyDrive/Mesh_segmentation_new/Testing_Results'
subject     = 'HC023'
# -------------------------------------------------------------------------

In [5]:
features_file = os.path.join(test_dir, f'{subject}_features_vertices.mat')
labels_file   = os.path.join(test_dir, f'{subject}_mesh_gt.mat')
os.makedirs(results_dir, exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [6]:
class MLP(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes, dropout=0.2):
        super(MLP, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.fc3 = nn.Linear(hidden_size, hidden_size)
        self.fc4 = nn.Linear(hidden_size, num_classes)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, x):
        x = self.dropout(self.relu(self.fc1(x)))
        x = self.dropout(self.relu(self.fc2(x)))
        x = self.dropout(self.relu(self.fc3(x)))
        return self.fc4(x)

In [7]:
fv = loadmat(features_file)['features_vertices']
gt = loadmat(labels_file)['Label_vertices']
n_surfaces = fv.size

pred_cells = np.empty((n_surfaces, 1), dtype=object)
prob_cells = np.empty((n_surfaces, 1), dtype=object)
idx_cells  = np.empty((n_surfaces, 1), dtype=object)
metrics    = []

In [8]:
for c in range(1, n_surfaces + 1):
    ckpt = torch.load(os.path.join(model_dir, f'mlp_cell{c:02d}.pt'),
                      map_location='cpu', weights_only=True)

    model = MLP(ckpt['input_size'], ckpt['hidden_size'],
                ckpt['num_classes'], ckpt['dropout'])
    model.load_state_dict(ckpt['state_dict'])
    model.to(device).eval()                         # dropout OFF

    X = np.asarray(fv.ravel()[c - 1], dtype=np.float64)
    y = np.zeros(X.shape[0], dtype=np.int64)
    y[np.asarray(gt.ravel()[c - 1]).ravel().astype(int) - 1] = 1

    Xs = (X - ckpt['scaler_mean'].numpy()) / ckpt['scaler_scale'].numpy()
    with torch.no_grad():
        logits = model(torch.tensor(Xs, dtype=torch.float32, device=device))
        prob = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
    pred = (prob >= ckpt['threshold']).astype(np.int64)

    tp = int(np.sum((pred == 1) & (y == 1)))
    fp = int(np.sum((pred == 1) & (y == 0)))
    fn = int(np.sum((pred == 0) & (y == 1)))
    tn = int(np.sum((pred == 0) & (y == 0)))
    acc  = (tp + tn) / (tp + tn + fp + fn)
    rec  = tp / max(tp + fn, 1)
    prec = tp / max(tp + fp, 1)
    fpr  = fp / max(fp + tn, 1)

    pred_cells[c - 1, 0] = pred.reshape(-1, 1)
    prob_cells[c - 1, 0] = prob.reshape(-1, 1)
    idx_cells[c - 1, 0]  = (np.flatnonzero(pred) + 1).reshape(-1, 1)   # MATLAB 1-based
    metrics.append([c, acc, rec, prec, fpr, tp, fp, fn, tn])

    print(f'[surface {c:2d}] acc {acc*100:5.2f}%  recall {rec*100:5.2f}%  '
          f'precision {prec*100:5.2f}%  FPR {fpr*100:5.2f}%  '
          f'TP {tp:5d}  FP {fp:6d}  FN {fn:5d}')

metrics = np.array(metrics, dtype=np.float64)
print('\nmean:  acc {:.2f}%  recall {:.2f}%  precision {:.2f}%  FPR {:.2f}%'
      .format(*(100 * metrics[:, 1:5].mean(axis=0))))

[surface  1] acc 84.42%  recall 41.94%  precision 59.67%  FPR  6.24%  TP  4082  FP   2759  FN  5650
[surface  2] acc 84.76%  recall 55.30%  precision 69.66%  FPR  6.86%  TP  7176  FP   3125  FN  5800
[surface  3] acc 85.63%  recall 67.90%  precision 71.97%  FPR  8.60%  TP  9836  FP   3831  FN  4651
[surface  4] acc 85.85%  recall 67.56%  precision 76.60%  FPR  7.50%  TP 10506  FP   3209  FN  5044
[surface  5] acc 86.90%  recall 72.50%  precision 76.35%  FPR  7.98%  TP 10907  FP   3379  FN  4137
[surface  6] acc 87.74%  recall 72.20%  precision 79.10%  FPR  6.76%  TP 10686  FP   2824  FN  4114
[surface  7] acc 89.35%  recall 73.92%  precision 81.16%  FPR  5.61%  TP 10177  FP   2362  FN  3590
[surface  8] acc 89.24%  recall 70.94%  precision 81.15%  FPR  5.10%  TP  9373  FP   2177  FN  3839
[surface  9] acc 89.14%  recall 70.21%  precision 80.24%  FPR  5.19%  TP  9098  FP   2241  FN  3861
[surface 10] acc 88.93%  recall 69.88%  precision 77.50%  FPR  5.71%  TP  8676  FP   2519  FN  3739


In [9]:
out_path = os.path.join(results_dir, f'{subject}_pred.mat')
savemat(out_path, {'predicted': pred_cells,
                   'predicted_prob': prob_cells,
                   'predicted_idx': idx_cells,
                   'metrics': metrics})
print('saved', out_path)

saved /content/drive/MyDrive/Mesh_segmentation_new/Testing_Results/HC023_pred.mat
